# 🚀 Cloud GPU Patent Embedder & FAISS Vector Indexer
### SDTD Multi-Agent Patent Intelligence System — Cloud Worker

Embeds **~120,000 Google Patents** (title + abstract) with Patent-BERT on Colab's free GPU and builds the search index used by the app.

**What this notebook does:**
1. Loads `index_records.parquet` made locally by `pull_google_patents.py` (Google Patents BigQuery: title, abstract, **IPC + CPC codes**).
2. Runs fast **FP16 batched GPU inference** with `anferico/bert-for-patents` (345M parameters).
3. Builds a compact **FAISS Inner-Product (cosine) index** (`patents_faiss.index`, 8-bit) and metadata table (`patents_meta.parquet`).
4. Saves both files to Google Drive and/or downloads them so you can drop them into the project folder.

⚡ **First**: Runtime → Change runtime type → Hardware accelerator → **T4 GPU**.

In [ ]:
# Step 1: Check GPU & Install Dependencies
!nvidia-smi
!pip install -q transformers torch pyarrow pandas tqdm faiss-cpu

In [ ]:
# Step 2: Storage Setup (Google Drive recommended: the output files are a few hundred MB)
import os

MOUNT_GOOGLE_DRIVE = True

OUTPUT_DIR = "/content"
if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        OUTPUT_DIR = '/content/drive/MyDrive/patent_faiss_index'
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        print(f"[*] Outputs will be saved to Google Drive: {OUTPUT_DIR}")
    except Exception as e:
        print(f"[*] Drive mount skipped ({e}). Using /content...")
        OUTPUT_DIR = "/content"
else:
    print(f"[*] Using Colab storage at: {OUTPUT_DIR}")

In [ ]:
# Step 3: Load the Google Patents records (title + abstract, IPC + CPC codes)
# Upload data/processed/index_records.parquet from the project folder when prompted,
# or put it in OUTPUT_DIR on Drive beforehand to skip the upload.
import pandas as pd
from google.colab import files

records_file = os.path.join(OUTPUT_DIR, "index_records.parquet")
if not os.path.exists(records_file):
    uploaded = files.upload()
    records_file = next(iter(uploaded))

df = pd.read_parquet(records_file)
records = []
for r in df.to_dict(orient="records"):
    for col in ("codes", "ipc_codes", "cpc_codes"):
        if col in r:
            r[col] = list(r[col])
    records.append(r)
texts = [r["meta"] for r in records]

print(f"Loaded {len(records):,} patents with columns {list(df.columns)}")
print("With IPC:", sum(1 for r in records if r.get("ipc_codes")), "| With CPC:", sum(1 for r in records if r.get("cpc_codes")))
print("Sample codes:", [r["codes"][:4] for r in records[:3]])

In [ ]:
# Step 4: Batched GPU Inference with Patent-BERT (FP16)
import torch
import numpy as np
from tqdm import tqdm
from transformers import BertModel, BertTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[*] Active compute device: {device}")
if device == "cuda":
    print(f"[*] GPU: {torch.cuda.get_device_name(0)}")

model_name = "anferico/bert-for-patents"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name).to(device)
if device == "cuda":
    model = model.half()
model.eval()

BATCH_SIZE = 128 if device == "cuda" else 16
# Embed in length order so each batch pads to similar lengths (much faster), then restore the order
order = np.argsort([len(t) for t in texts])
embeddings = np.zeros((len(texts), model.config.hidden_size), dtype=np.float32)

print(f"Embedding {len(texts):,} patents in batches of {BATCH_SIZE}...")
with torch.no_grad():
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="BERT GPU Forward Pass"):
        idx = order[i : i + BATCH_SIZE]
        tok = tokenizer([texts[j] for j in idx], return_tensors="pt", padding=True,
                        truncation=True, max_length=512).to(device)
        out = model(**tok)
        # Mean pooling across tokens (same as the app's RetrievalAgent)
        pooled = torch.mean(out.last_hidden_state, dim=1)
        embeddings[idx] = pooled.float().cpu().numpy()

print(f"[*] Embedding matrix: {embeddings.shape}")

In [ ]:
# Step 5: Build the FAISS index (8-bit scalar quantized: ~4x smaller, near-identical cosine scores)
import faiss

faiss.normalize_L2(embeddings)
dim = embeddings.shape[1]
index = faiss.IndexScalarQuantizer(dim, faiss.ScalarQuantizer.QT_8bit, faiss.METRIC_INNER_PRODUCT)
index.train(embeddings)
index.add(embeddings)

index_file = os.path.join(OUTPUT_DIR, "patents_faiss.index")
meta_file = os.path.join(OUTPUT_DIR, "patents_meta.parquet")
faiss.write_index(index, index_file)
pd.DataFrame(records).to_parquet(meta_file, index=False)

print("=" * 60)
print(f"✅ FAISS index with {index.ntotal:,} patent vectors")
print(f"1. Index: {index_file} ({os.path.getsize(index_file)/(1024*1024):.1f} MB)")
print(f"2. Meta:  {meta_file} ({os.path.getsize(meta_file)/(1024*1024):.1f} MB)")
print("=" * 60)

In [ ]:
# Step 6: Quick In-Colab Test Query
sample_query = "A semiconductor cooling system with phase change heat sink and micro-channels"
print(f"Testing retrieval with query: '{sample_query}'")

with torch.no_grad():
    q_tok = tokenizer([sample_query], return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    q_emb = torch.mean(model(**q_tok).last_hidden_state, dim=1).float().cpu().numpy().astype(np.float32)
    faiss.normalize_L2(q_emb)

scores, indices = index.search(q_emb, 3)
print("\n--- Top 3 Retrieved Patents ---")
for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), 1):
    rec = records[idx]
    print(f"#{rank} [Score: {score:.4f}] Patent ID: {rec['patent_id']} | IPC: {rec['codes']}")
    print(f"   {rec['meta'][:150]}...\n")

In [ ]:
# Step 7: Get the files onto your PC
# If you used Google Drive (Step 2), download them from MyDrive/patent_faiss_index instead;
# browser downloads of large files from Colab can fail.
from google.colab import files

if not OUTPUT_DIR.startswith("/content/drive"):
    files.download(index_file)
    files.download(meta_file)
print("Put both files in the project root folder (replace the old ones):")
print("  -> patents_faiss.index")
print("  -> patents_meta.parquet")